# NumCompute-Stream Demo
This notebook demonstrates chunk-wise streaming training.

In [ ]:
from numcompute_stream.io import load_csv, make_stream_batches
from numcompute_stream.preprocessing import StandardScaler
from numcompute_stream.tree import DecisionTreeClassifier
from numcompute_stream.ensemble import EnsembleClassifier
from numcompute_stream.pipeline import Pipeline
from numcompute_stream.stream import StreamTrainer
from numcompute_stream.visualise import plot_metric_over_time, compare_models


In [ ]:
X, y = load_csv('sample_stream_data.csv', target_column='target', header=True)
chunks = make_stream_batches(X, y, chunk_size=8)
print(X.shape, y.shape, len(chunks))


In [ ]:
tree_pipe = Pipeline([("scale", StandardScaler()), ("tree", DecisionTreeClassifier(max_depth=4, random_state=3))])
forest_pipe = Pipeline([("scale", StandardScaler()), ("forest", EnsembleClassifier(n_estimators=5, max_depth=4, random_state=3))])


In [ ]:
tree_runner = StreamTrainer(tree_pipe)
forest_runner = StreamTrainer(forest_pipe)
tree_scores, forest_scores = [], []
for X_chunk, y_chunk in chunks:
    tree_runner.fit_chunk(X_chunk, y_chunk)
    tree_scores.append(tree_runner.score_chunk(X_chunk, y_chunk))
    forest_runner.fit_chunk(X_chunk, y_chunk)
    forest_scores.append(forest_runner.score_chunk(X_chunk, y_chunk))
print(tree_scores)
print(forest_scores)


In [ ]:
plot_metric_over_time(tree_scores, title='Decision Tree Accuracy Over Chunks', ylabel='Accuracy', show=True)
compare_models(tree_scores, forest_scores, labels=('Tree', 'Forest'), show=True)
